In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import joblib

In [2]:
# ============================================================
# Load Merged Dataset
# ============================================================

ds = xr.open_dataset("merged_glory_chla_colocated_100-Copy3.nc")

print(ds)

<xarray.Dataset>
Dimensions:      (time: 339, latitude: 337, longitude: 661)
Coordinates:
  * latitude     (latitude) float32 0.0 0.08333 0.1667 0.25 ... 27.83 27.92 28.0
  * longitude    (longitude) float32 45.0 45.08 45.17 ... 99.83 99.92 100.0
  * time         (time) datetime64[ns] 1997-10-01 1997-11-01 ... 2025-12-01
Data variables:
    zos          (time, latitude, longitude) float64 ...
    so           (time, latitude, longitude) float64 ...
    thetao       (time, latitude, longitude) float64 ...
    uo           (time, latitude, longitude) float64 ...
    vo           (time, latitude, longitude) float64 ...
    mlotst       (time, latitude, longitude) float64 ...
    vorticity    (time, latitude, longitude) float64 ...
    okubo_weiss  (time, latitude, longitude) float64 ...
    grad_ssh     (time, latitude, longitude) float64 ...
    chlor_a      (time, latitude, longitude) float32 ...
Attributes:
    history:                   2023/06/01 16:20:05 MERCATOR OCEAN Netcdf crea..

In [3]:
# ============================================================
#  Regional coordinates
# ============================================================

ds = ds.sel(
    latitude=slice(5, 25),
    longitude=slice(78, 100)                # check properly the regional coordinates
)

print(ds)


print(ds.latitude.min().values)
print(ds.latitude.max().values)

print(ds.longitude.min().values)
print(ds.longitude.max().values)

<xarray.Dataset>
Dimensions:      (time: 339, latitude: 241, longitude: 265)
Coordinates:
  * latitude     (latitude) float32 5.0 5.083 5.167 5.25 ... 24.83 24.92 25.0
  * longitude    (longitude) float32 78.0 78.08 78.17 ... 99.83 99.92 100.0
  * time         (time) datetime64[ns] 1997-10-01 1997-11-01 ... 2025-12-01
Data variables:
    zos          (time, latitude, longitude) float64 ...
    so           (time, latitude, longitude) float64 ...
    thetao       (time, latitude, longitude) float64 ...
    uo           (time, latitude, longitude) float64 ...
    vo           (time, latitude, longitude) float64 ...
    mlotst       (time, latitude, longitude) float64 ...
    vorticity    (time, latitude, longitude) float64 ...
    okubo_weiss  (time, latitude, longitude) float64 ...
    grad_ssh     (time, latitude, longitude) float64 ...
    chlor_a      (time, latitude, longitude) float32 ...
Attributes:
    history:                   2023/06/01 16:20:05 MERCATOR OCEAN Netcdf crea...
 

In [4]:
# ============================================================
# Select 1998-2025
# ============================================================

ds_pred = ds.sel(
    time=slice(
        "1998-01-01",
        "2025-12-31"
    )
)

print(ds_pred)
print(ds_pred.time.values)

<xarray.Dataset>
Dimensions:      (time: 336, latitude: 241, longitude: 265)
Coordinates:
  * latitude     (latitude) float32 5.0 5.083 5.167 5.25 ... 24.83 24.92 25.0
  * longitude    (longitude) float32 78.0 78.08 78.17 ... 99.83 99.92 100.0
  * time         (time) datetime64[ns] 1998-01-01 1998-02-01 ... 2025-12-01
Data variables:
    zos          (time, latitude, longitude) float64 ...
    so           (time, latitude, longitude) float64 ...
    thetao       (time, latitude, longitude) float64 ...
    uo           (time, latitude, longitude) float64 ...
    vo           (time, latitude, longitude) float64 ...
    mlotst       (time, latitude, longitude) float64 ...
    vorticity    (time, latitude, longitude) float64 ...
    okubo_weiss  (time, latitude, longitude) float64 ...
    grad_ssh     (time, latitude, longitude) float64 ...
    chlor_a      (time, latitude, longitude) float32 ...
Attributes:
    history:                   2023/06/01 16:20:05 MERCATOR OCEAN Netcdf crea...
 

In [5]:
# ============================================================
# Log10 Chlorophyll
# ============================================================

ds_pred["chlor_a_log10"] = xr.where(
    ds_pred["chlor_a"] > 0,
    np.log10(ds_pred["chlor_a"]),
    np.nan
)


# ============================================================
# Natural Log MLD
# ============================================================

ds_pred["mlotst_ln"] = xr.where(
    ds_pred["mlotst"] > 0,
    np.log(ds_pred["mlotst"]),
    np.nan
)

In [6]:
# ============================================================
# Month Feature
# ============================================================

month = ds_pred.time.dt.month

# ============================================================
# Cyclic Month Features
# ============================================================

ds_pred["month_sin"] = xr.DataArray(
    np.sin(2 * np.pi * month / 12),
    coords={"time": ds_pred.time},
    dims=["time"]
)

ds_pred["month_cos"] = xr.DataArray(
    np.cos(2 * np.pi * month / 12),
    coords={"time": ds_pred.time},
    dims=["time"]
)

In [7]:
# ============================================================
# Expand Month Features
# ============================================================

month_sin_3d = ds_pred["month_sin"].broadcast_like(
    ds_pred["so"]
)

month_cos_3d = ds_pred["month_cos"].broadcast_like(
    ds_pred["so"]
)

In [8]:
# ============================================================
# Feature Stack
# ============================================================

feature_stack = np.stack(
    [
        month_sin_3d.values,
        month_cos_3d.values,
        #ds_pred["zos"].values,
        ds_pred["so"].values,
        ds_pred["thetao"].values,
        ds_pred["uo"].values,
        ds_pred["vo"].values,
        ds_pred["mlotst_ln"].values,
        ds_pred["vorticity"].values,
        ds_pred["okubo_weiss"].values
        #ds_pred["grad_ssh"].values
    ],
    axis=-1
)

print(feature_stack.shape)

(336, 241, 265, 11)


In [9]:
# ============================================================
# Valid Ocean Mask
# ============================================================

valid_mask = np.isfinite(
    feature_stack
).all(axis=-1)

print(valid_mask.shape)

(336, 241, 265)


In [10]:
FEATURES = [
    "month_sin",
    "month_cos",
    #"zos",
    "so",
    "thetao",
    "uo",
    "vo",
    "mlotst_ln",
    "vorticity",
    "okubo_weiss"
    #"grad_ssh"
]

X_pred = pd.DataFrame(
    feature_stack[valid_mask],
    columns=FEATURES
).astype("float32")

In [11]:
UNDER_PENALTY = 0.007                             # check for the correct value of under-penalty region and model wise 

def underestimation_loss(y_true, y_pred):

    error = y_true - y_pred

    under_mask = error > 0

    grad = np.where(
        under_mask,
        -2.0 * (1.0 + UNDER_PENALTY) * error,
        -2.0 * error
    )

    hess = np.where(
        under_mask,
        2.0 * (1.0 + UNDER_PENALTY),
        2.0
    )

    return grad, hess

In [12]:
# ============================================================
# Load Trained LGBM Model
# ============================================================

import joblib

model = joblib.load(
    "xgboost_BoB_Regu_model_4.pkl"                   # give the best trained model, to be incorporated to make the long term data
)

In [13]:
# ============================================================
# Prediction
# ============================================================

pred_log10 = model.predict(
    X_pred
)

print(pred_log10.shape)

(12485424,)


In [14]:
# ============================================================
# Rebuild Grid
# ============================================================

pred_grid_log10 = np.full(
    valid_mask.shape,
    np.nan,
    dtype=np.float32
)

pred_grid_log10[
    valid_mask
] = pred_log10

In [15]:
# ============================================================
# Back Transform
# ============================================================

pred_grid_chl = np.power(
    10,
    pred_grid_log10
)

In [16]:
# ============================================================
# Add Predictions
# ============================================================

ds_pred["chlor_a_pred_log10"] = (
    ("time", "latitude", "longitude"),
    pred_grid_log10
)

ds_pred["chlor_a_pred"] = (
    ("time", "latitude", "longitude"),
    pred_grid_chl
)

In [17]:
# ============================================================
# Error
# ============================================================

ds_pred["prediction_error"] = (
    ds_pred["chlor_a_pred"]
    -
    ds_pred["chlor_a"]
)

In [18]:
# ============================================================
# Create Output Dataset
# ============================================================

output_ds = xr.Dataset(

    coords={

        "time": ds_pred.time,
        "latitude": ds_pred.latitude,
        "longitude": ds_pred.longitude
    }
)

# ============================================================
# Anti-log Chlorophyll Predictions
# ============================================================

output_ds["chlor_a_pred"] = (

    ("time", "latitude", "longitude"),
    pred_grid_chl
)

# ============================================================
# Save NetCDF
# ============================================================

OUTPUT_FILE = (
    "XGBOOST_Bay_Of_Bengal_predictions_1998_2025.nc"                 # output file name to be checked
)

output_ds.to_netcdf(
    OUTPUT_FILE
)

print(
    f"\nSaved successfully: "
    f"{OUTPUT_FILE}"
)


Saved successfully: XGBOOST_Bay_Of_Bengal_predictions_1998_2025.nc


In [19]:
# ============================================================
# Load Merged Dataset
# ============================================================

ds = xr.open_dataset("merged_glory_chla_colocated_100-Copy3.nc")       # check the dataset and its path properly
ds = ds.sel(
    latitude=slice(5, 28),
    longitude=slice(45, 65)
)

ds_pred = ds.sel(
    time=slice(
        "1998-01-01",
        "2025-12-31"
    )
)

print(ds_pred)



<xarray.Dataset>
Dimensions:      (time: 336, latitude: 277, longitude: 241)
Coordinates:
  * latitude     (latitude) float32 5.0 5.083 5.167 5.25 ... 27.83 27.92 28.0
  * longitude    (longitude) float32 45.0 45.08 45.17 45.25 ... 64.83 64.92 65.0
  * time         (time) datetime64[ns] 1998-01-01 1998-02-01 ... 2025-12-01
Data variables:
    zos          (time, latitude, longitude) float64 ...
    so           (time, latitude, longitude) float64 ...
    thetao       (time, latitude, longitude) float64 ...
    uo           (time, latitude, longitude) float64 ...
    vo           (time, latitude, longitude) float64 ...
    mlotst       (time, latitude, longitude) float64 ...
    vorticity    (time, latitude, longitude) float64 ...
    okubo_weiss  (time, latitude, longitude) float64 ...
    grad_ssh     (time, latitude, longitude) float64 ...
    chlor_a      (time, latitude, longitude) float32 ...
Attributes:
    history:                   2023/06/01 16:20:05 MERCATOR OCEAN Netcdf crea

In [20]:
# ============================================================
# Load Merged Dataset
# ============================================================

ds = xr.open_dataset("XGBOOST_Bay_Of_Bengal_predictions_1998_2025.nc")      # Rename as required
 
print(ds)

<xarray.Dataset>
Dimensions:       (time: 336, latitude: 241, longitude: 265)
Coordinates:
  * time          (time) datetime64[ns] 1998-01-01 1998-02-01 ... 2025-12-01
  * latitude      (latitude) float32 5.0 5.083 5.167 5.25 ... 24.83 24.92 25.0
  * longitude     (longitude) float32 78.0 78.08 78.17 ... 99.83 99.92 100.0
Data variables:
    chlor_a_pred  (time, latitude, longitude) float32 ...
